<a href="https://colab.research.google.com/github/agastyabisht/AgenticAI/blob/main/TransformarArch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai tiktoken matplotlib seaborn numpy

In [ ]:
import os
import getpass
from openai import OpenAI

# Input your OpenAI API key securely (or assign it directly: OPENAI_APIKEY = "sk-...")
OPENAI_APIKEY = os.environ.get("OPENAI_API_KEY") or getpass.getpass("Enter your OpenAI API Key: ")

# Initialize OpenAI client
client = OpenAI(api_key=OPENAI_APIKEY)

In [ ]:
# This shows how LLM takes input and how it Process - Transformar Architecture
# First it converts the words into token
# Than it converts those token into embedding - means to vendor data - Each token into 786 numeric values. These numeric values are given based on trained data
# Like if 2 words are similar than there ventor data will have similarities - and if we plot it into a graph - we can see that similar meaning (not similar wording) words graph
# is close to one another.

import numpy as np
import matplotlib.pyplot as plt
import tiktoken

# -------------------------------------------------------------------
# 1. INPUT WORDS
# -------------------------------------------------------------------
word1 = "happy"
word2 = "joyful"      # Semantically similar to word1
word3 = "Angry"    # Completely unrelated word for comparison


# -------------------------------------------------------------------
# 2. TOKENIZATION (Converting Words to Tokens)
# -------------------------------------------------------------------
# Load the tokenizer used by OpenAI models (cl100k_base)
encoding = tiktoken.get_encoding("cl100k_base")

tokens_1 = encoding.encode(word1)
tokens_2 = encoding.encode(word2)
tokens_3 = encoding.encode(word3)

print("--- TOKENIZATION RESULTS ---")
print(f"'{word1}' -> Token ID(s): {tokens_1}")
print(f"'{word2}' -> Token ID(s): {tokens_2}")
print(f"'{word3}' -> Token ID(s): {tokens_3}\n")


# -------------------------------------------------------------------
# 3. EMBEDDING GENERATION (Converting Tokens to Numeric Vectors)
# -------------------------------------------------------------------
# We request a 768-dimension embedding vector
MODEL_NAME = "text-embedding-3-small"
TARGET_DIMENSIONS = 768

response = client.embeddings.create(
    input=[word1, word2, word3],
    model=MODEL_NAME,
    dimensions=TARGET_DIMENSIONS
)

# Extract embedding arrays
emb1 = np.array(response.data[0].embedding)
emb2 = np.array(response.data[1].embedding)
emb3 = np.array(response.data[2].embedding)

print("--- EMBEDDING GENERATION ---")
print(f"Vector dimension count: {len(emb1)} floating-point numbers")
print(f"First 5 numbers for '{word1}': {emb1[:5].round(4)}")
print(f"First 5 numbers for '{word2}': {emb2[:5].round(4)}\n")


# -------------------------------------------------------------------
# 4. COSINE SIMILARITY MATH
# -------------------------------------------------------------------
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_1_2 = cosine_similarity(emb1, emb2)
sim_1_3 = cosine_similarity(emb1, emb3)

print("--- COSINE SIMILARITY SCORES ---")
print(f"Similarity ('{word1}' vs '{word2}'): {sim_1_2:.4f} (High -> Similar meaning)")
print(f"Similarity ('{word1}' vs '{word3}'): {sim_1_3:.4f} (Lower -> Different meaning)\n")


# -------------------------------------------------------------------
# 5. GRAPH PLOTTING
# -------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# --- Plot A: Line Pattern Comparison (First 100 Dimensions) ---
# Slicing first 100 dimensions so peaks and valleys are visually clear
subset = 100
dim_range = np.arange(subset)

ax1.plot(dim_range, emb1[:subset], label=f"'{word1}'", color='#1f77b4', linewidth=2, alpha=0.9)
ax1.plot(dim_range, emb2[:subset], label=f"'{word2}' (Similar)", color='#2ca02c', linewidth=2, alpha=0.9)
ax1.plot(dim_range, emb3[:subset], label=f"'{word3}' (Unrelated)", color='#d62728', linewidth=1, linestyle='--', alpha=0.5)

ax1.set_title(f"1. Vector Pattern Overlay across Dimensions (First {subset} of {TARGET_DIMENSIONS})", fontsize=13, fontweight='bold')
ax1.set_xlabel("Vector Dimension Index", fontsize=11)
ax1.set_ylabel("Numerical Value", fontsize=11)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(loc='upper right', fontsize=11)

# --- Plot B: Dimension Correlation Scatter ---
ax2.scatter(emb1, emb2, color='green', alpha=0.4, label=f"'{word1}' vs '{word2}' (Similar: {sim_1_2:.2f})")
ax2.scatter(emb1, emb3, color='red', alpha=0.2, label=f"'{word1}' vs '{word3}' (Unrelated: {sim_1_3:.2f})")

ax2.set_title("2. Dimension-by-Dimension Correlation Scatter Plot", fontsize=13, fontweight='bold')
ax2.set_xlabel(f"Vector Values of '{word1}'", fontsize=11)
ax2.set_ylabel("Vector Values of Target Word", fontsize=11)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(loc='upper left', fontsize=11)

plt.tight_layout()
plt.show()